In [1]:
import pandas as pd
crime_df = pd.read_csv("Data Files/crime_dataset_india.csv")
print(len(crime_df))
print(crime_df.columns)

40160
Index(['Report Number', 'Date Reported', 'Date of Occurrence',
       'Time of Occurrence', 'City', 'Crime Code', 'Crime Description',
       'Victim Age', 'Victim Gender', 'Weapon Used', 'Crime Domain',
       'Police Deployed', 'Case Closed', 'Date Case Closed'],
      dtype='object')


In [2]:
#Converting crime dataframe to a monthly time series dataframe
"""
I used monthly aggregation to reduce noise and build a stable time-series model, since raw-incident level data
is unsuitable for forecasting
"""

crime_df["Date of Occurrence"] = pd.to_datetime(crime_df["Date of Occurrence"])
crime_df["Year of Occurrence"] = crime_df["Date of Occurrence"].dt.year
crime_df["Month of Occurrence"] = crime_df["Date of Occurrence"].dt.month

monthly_df = (crime_df.groupby(["City", "Year of Occurrence", "Month of Occurrence"]).size().reset_index(name="Crime_Count"))
print(monthly_df)


               City  Year of Occurrence  Month of Occurrence  Crime_Count
0              Agra                2020                    1           11
1              Agra                2020                    2            7
2              Agra                2020                    3           19
3              Agra                2020                    4            9
4              Agra                2020                    5           20
...             ...                 ...                  ...          ...
1590  Visakhapatnam                2024                    3           18
1591  Visakhapatnam                2024                    4           10
1592  Visakhapatnam                2024                    5           21
1593  Visakhapatnam                2024                    6           16
1594  Visakhapatnam                2024                    7           19

[1595 rows x 4 columns]


In [3]:
#Creating a continuous time column
monthly_df["Date"] = pd.to_datetime(
                    monthly_df["Year of Occurrence"].astype(str) + "-" +
                    monthly_df["Month of Occurrence"].astype(str) + "-01"
)
monthly_df[["Year of Occurrence", "Month of Occurrence", "Date"]].head()

,Year of Occurrence,Month of Occurrence,Date
0,2020,1,2020-01-01
1,2020,2,2020-02-01
2,2020,3,2020-03-01
3,2020,4,2020-04-01
4,2020,5,2020-05-01


In [ ]:
"""
The following code adds months that would otherwise have been removed because of the crime count in 
that specific month and city being 0. This code is redundant because none of the rows in the monthly_df
dataframe have a 0 value for crime count. I have still included it for datasets that would have required it
"""
def complete_months(city_df):
    city_df = city_df.set_index("Date").sort_index()
    full_range = pd.date_range(
        start = city_df.index.min(),
        end = city_df.index.max(),
        freq = "MS" #Groups by month start
    )
    return city_df.reindex(full_range, fill_value = 0).rename_axis("Date").reset_index()

monthly_complete = (
    monthly_df
    .groupby("City", group_keys=False)
    .apply(complete_months)
)
print(len(monthly_df) == len(monthly_complete))
print(monthly_complete.head())

True
        Date  City  Year of Occurrence  Month of Occurrence  Crime_Count
0 2020-01-01  Agra                2020                    1           11
1 2020-02-01  Agra                2020                    2            7
2 2020-03-01  Agra                2020                    3           19
3 2020-04-01  Agra                2020                    4            9
4 2020-05-01  Agra                2020                    5           20


C:\Users\abum0\AppData\Local\Temp\ipykernel_36028\2760956916.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(complete_months)


In [5]:
#Sorting the months and cities
monthly_complete = monthly_complete.sort_values(["City", "Date"])

"""
Raw incident-level crime data was transformed into a monthly time series by aggregating records 
city-wise and month-wise. Missing months were explicitly handled to ensure temporal continuity.
"""

'\nRaw incident-level crime data was transformed into a monthly time series by aggregating records \ncity-wise and month-wise. Missing months were explicitly handled to ensure temporal continuity.\n'

In [6]:
#Turning it into a csv so it can be accessed anywhere
monthly_complete.to_csv("monthly_data.csv", index = False)